# Local Gherkin Case Generation in JSON — RunPod

This notebook was adapted for execution on **RunPod/Jupyter**.

It:

1. receives a CSV or XLSX spreadsheet available in the RunPod environment;
2. uses only the `id` and `name` columns;
3. downloads and runs a Hugging Face model locally on the Pod GPU;
4. sequentially executes the three techniques: **zero-shot**, **one-shot**, and **few-shot**;
5. runs each case the configured number of times;
6. generates **an independent JSON file for each technique**;
7. saves progress after each case, allowing an interrupted execution to be resumed;
8. keeps the model loaded only once on the GPU throughout the three techniques;
9. builds the JSON in the code, without asking the model to produce JSON.

Generated files:

- `geracoes_gherkin_<modelo>_zero-shot.json`
- `geracoes_gherkin_<modelo>_one-shot.json`
- `geracoes_gherkin_<modelo>_few-shot.json`

The Zero-shot, One-shot, and Few-shot prompts were preserved exactly as in the original notebook.


## 1. Dependency Installation


In [ ]:
%pip install -q -U "transformers>=4.45.0" accelerate bitsandbytes safetensors sentencepiece huggingface_hub pandas openpyxl tqdm

## 2. Imports and GPU Check


In [ ]:
import os
import re
import json
import time
from pathlib import Path
from typing import Optional

import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = "cuda"

    print("GPU:", torch.cuda.get_device_name(0))

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    )

    print(f"Total GPU memory: {total_memory:.1f} GB")

else:
    DEVICE = "cpu"

    print("CUDA GPU not available.")
    print("The notebook will continue execution using the CPU.")

print("Selected device:", DEVICE)

## 3. Spreadsheet Location on RunPod


In [ ]:
# On RunPod, upload the spreadsheet using the Jupyter file browser.
# The Pod's persistent directory is usually /workspace.
#
# Example:
#   /workspace/casos.csv
#   /workspace/casos.xlsx

INPUT_FILE = "/workspace/casos.csv"

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"File not found: {INPUT_FILE}\n"
        "Upload the spreadsheet through Jupyter and set INPUT_FILE to the correct path."
    )

print("Selected file:", INPUT_FILE)


## 4. Experiment Configuration


In [ ]:
# ------------------------------------------------------------------
# LOCAL MODEL
# ------------------------------------------------------------------

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

# This can also be a previously downloaded local path:
# MODEL_ID = "/workspace/models/my_model"

# ------------------------------------------------------------------
# EXPERIMENT
# ------------------------------------------------------------------

# The three techniques will be executed in this order.
TECHNIQUES = [
    "zero-shot",
    "one-shot",
    "few-shot",
]

# Number of times EACH case will be sent to the model FOR EACH TECHNIQUE.
NUM_EXECUTIONS = 10

# None processes all cases in the spreadsheet.
# Use 200 to process only the first 200 cases.
NUM_CASES = None

# Generation parameters.
TEMPERATURE = 0.5
TOP_P = 0.8
MAX_NEW_TOKENS = 256
MAX_RETRIES = 3
BASE_SEED = 42

# Local quantization.
USE_4BIT = True
TRUST_REMOTE_CODE = False

# Persistent output directory on RunPod.
OUTPUT_DIR = Path("/workspace/resultados_gherkin")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if NUM_EXECUTIONS < 1:
    raise ValueError("NUM_EXECUTIONS must be greater than or equal to 1.")

if NUM_CASES is not None and NUM_CASES < 1:
    raise ValueError("NUM_CASES must be None or greater than or equal to 1.")

print("Techniques:", ", ".join(TECHNIQUES))
print("Output directory:", OUTPUT_DIR)


## 5. Zero-shot Prompt


In [ ]:
ZERO_SHOT_PROMPT = """
Converta a seguinte descrição de caso de teste em um único cenário BDD, usando a sintaxe estrita de Gherkin. Certifique-se de que a saída contenha apenas a sintaxe de Gherkin para o cenário, sem comentários, explicações ou a palavra "Feature". Use o português para as descrições dos casos de teste e detalhes do cenário, mas mantenha as palavras-chave do Gherkin em inglês.

Agora, converta a seguinte descrição de caso de teste em exatamente um cenário BDD usando a sintaxe estrita de Gherkin. A saída deve seguir exatamente o formato do exemplo fornecido e não conter nada além do cenário BDD. Somente as palavras-chave do Gherkin devem estar em inglês; todo o outro texto deve estar em português. Se o caso de teste fornecido estiver em inglês, traduza-o para o português na geração do cenário BDD, mantendo as palavras-chave (Given, When, Then, And) em inglês.

Descrição do Caso de Teste:
{test_case}

Para um bom cenário BDD, certifique-se de declarar claramente o valor de negócio ou resultado esperado e mantenha o foco em uma única ação e seu resultado. Use apenas os passos essenciais (Given, When, Then, And) de forma clara e declarativa, evitando detalhes de implementação e repetições desnecessárias. Garanta que os cenários sejam independentes, utilizem uma terminologia de negócios consistente sem jargões técnicos e que os passos sejam escritos em terceira pessoa para evitar múltiplas interpretações.

Certifique-se de que os cenários BDD tenham indentação consistente de dois espaços para cada passo sob 'Scenario', sem linhas em branco entre os passos, e uma linha em branco separando diferentes cenários.

Siga esta estrutura para a saída:

Scenario: [Descrição do Cenário]
  Given [algum contexto inicial]
    And [mais algum contexto, se houver]
  When [uma ação é realizada]
  Then [um conjunto específico de resultados deve ocorrer]
    And [outro resultado, se houver]

A resposta deve conter estritamente apenas a sintaxe válida de Gherkin e evitar qualquer informação ou comentário extra. A resposta deve conter apenas o texto BDD, sem formatação ou texto adicional (por exemplo, sem 'gherkin```'), e deve ser apenas um cenário BDD.
""".strip()

## 6. One-shot Prompt


In [ ]:
ONE_SHOT_PROMPT = """
Converta a seguinte descrição de caso de teste em um único cenário BDD, usando a sintaxe estrita de Gherkin. Certifique-se de que a saída contenha apenas a sintaxe de Gherkin para o cenário, sem comentários, explicações ou a palavra "Feature". Use o português para as descrições dos casos de teste e detalhes do cenário, mas mantenha as palavras-chave do Gherkin em inglês.

# Exemplo
# Descrição do caso de teste:
# Usuário tenta login com credenciais inválidas.

Scenario: Login com senha inválida
  Given o usuário está na página de login
    And o usuário insere um nome de usuário válido
  When o usuário insere uma senha inválida e clica no botão de login
  Then o sistema exibe uma mensagem de erro indicando que a senha está incorreta
    And o campo de senha é limpo

Agora, converta a seguinte descrição de caso de teste em exatamente um cenário BDD usando a sintaxe estrita de Gherkin. A saída deve seguir exatamente o formato do exemplo fornecido e não conter nada além do cenário BDD. Somente as palavras-chave do Gherkin devem estar em inglês; todo o outro texto deve estar em português. Se o caso de teste fornecido estiver em inglês, traduza-o para o português na geração do cenário BDD, mantendo as palavras-chave (Given, When, Then, And) em inglês.

Descrição do Caso de Teste:
{test_case}

Para um bom cenário BDD, certifique-se de declarar claramente o valor de negócio ou resultado esperado e mantenha o foco em uma única ação e seu resultado. Use apenas os passos essenciais (Given, When, Then, And) de forma clara e declarativa, evitando detalhes de implementação e repetições desnecessárias. Garanta que os cenários sejam independentes, utilizem uma terminologia de negócios consistente sem jargões técnicos e que os passos sejam escritos em terceira pessoa para evitar múltiplas interpretações.
Certifique-se de que os cenários BDD tenham indentação consistente de dois espaços para cada passo sob 'Scenario', sem linhas em branco entre os passos, e uma linha em branco separando diferentes cenários.

Siga esta estrutura para a saída:

Scenario: [Descrição do Cenário]
  Given [algum contexto inicial]
    And [mais algum contexto, se houver]
  When [uma ação é realizada]
  Then [um conjunto específico de resultados deve ocorrer]
    And [outro resultado, se houver]

A resposta deve conter estritamente apenas a sintaxe válida de Gherkin e evitar qualquer informação ou comentário extra. A resposta deve conter apenas o texto BDD, sem formatação ou texto adicional (por exemplo, sem 'gherkin```'), e deve ser apenas um cenário BDD.
""".strip()

## 7. Few-shot Prompt


In [ ]:
FEW_SHOT_PROMPT = """
Converta a seguinte descrição de caso de teste em um único cenário BDD, usando a sintaxe estrita de Gherkin. Certifique-se de que a saída contenha apenas a sintaxe de Gherkin para o cenário, sem comentários, explicações ou a palavra "Feature". Use o português para as descrições dos casos de teste e detalhes do cenário, mas mantenha as palavras-chave do Gherkin em inglês.

# Exemplo 1
# Descrição do caso de teste:
# Usuário tenta login com credenciais inválidas.

Scenario: Login com senha inválida
  Given o usuário está na página de login
    And o usuário insere um nome de usuário válido
  When o usuário insere uma senha inválida e clica no botão de login
  Then o sistema exibe uma mensagem de erro indicando que a senha está incorreta
    And o campo de senha é limpo

# Exemplo 2
# Descrição do caso de teste:
# Usuário tenta redefinir a senha esquecida.

Scenario: Redefinição de senha com e-mail válido
  Given o usuário está na página de redefinição de senha
    And o usuário insere um endereço de e-mail registrado
  When o usuário clica no botão de enviar
  Then o sistema exibe uma mensagem indicando que um link de redefinição de senha foi enviado para o e-mail do usuário
    And o usuário é redirecionado para a página de login

# Exemplo 3
# Descrição do caso de teste:
# Usuário adiciona um item ao carrinho de compras.

Scenario: Adicionar item ao carrinho de compras
  Given o usuário está na página de detalhes de um produto
    And o produto está disponível em estoque
  When o usuário clica no botão "Adicionar ao carrinho"
  Then o item é adicionado ao carrinho de compras
    And o sistema exibe uma mensagem de confirmação "Item adicionado ao carrinho com sucesso"
    And o ícone do carrinho de compras é atualizado para refletir o novo item

Agora, converta a seguinte descrição de caso de teste em exatamente um cenário BDD usando a sintaxe estrita de Gherkin. A saída deve seguir exatamente o formato do exemplo fornecido e não conter nada além do cenário BDD. Somente as palavras-chave do Gherkin devem estar em inglês; todo o outro texto deve estar em português. Se o caso de teste fornecido estiver em inglês, traduza-o para o português na geração do cenário BDD, mantendo as palavras-chave (Given, When, Then, And) em inglês.

Descrição do Caso de Teste:
{test_case}

Para um bom cenário BDD, certifique-se de declarar claramente o valor de negócio ou resultado esperado e mantenha o foco em uma única ação e seu resultado. Use apenas os passos essenciais (Given, When, Then, And) de forma clara e declarativa, evitando detalhes de implementação e repetições desnecessárias. Garanta que os cenários sejam independentes, utilizem uma terminologia de negócios consistente sem jargões técnicos e que os passos sejam escritos em terceira pessoa para evitar múltiplas interpretações.
Certifique-se de que os cenários BDD tenham indentação consistente de dois espaços para cada passo sob 'Scenario', sem linhas em branco entre os passos, e uma linha em branco separando diferentes cenários.

Siga esta estrutura para a saída:

Scenario: [Descrição do Cenário]
  Given [algum contexto inicial]
    And [mais algum contexto, se houver]
  When [uma ação é realizada]
  Then [um conjunto específico de resultados deve ocorrer]
    And [outro resultado, se houver]

A resposta deve conter estritamente apenas a sintaxe válida de Gherkin e evitar qualquer informação ou comentário extra. A resposta deve conter apenas o texto BDD, sem formatação ou texto adicional (por exemplo, sem 'gherkin```'), e deve ser apenas um cenário BDD.
""".strip()

## 8. Prompt Registry


In [ ]:
TECHNIQUE_ALIASES = {
    "zero": "zero-shot",
    "zero-shot": "zero-shot",
    "zero_shot": "zero-shot",
    "one": "one-shot",
    "one-shot": "one-shot",
    "one_shot": "one-shot",
    "few": "few-shot",
    "few-shot": "few-shot",
    "few_shot": "few-shot",
}

PROMPTS = {
    "zero-shot": ZERO_SHOT_PROMPT,
    "one-shot": ONE_SHOT_PROMPT,
    "few-shot": FEW_SHOT_PROMPT,
}

normalized_techniques = []

for technique in TECHNIQUES:
    normalized_input = str(technique).strip().lower()

    if normalized_input not in TECHNIQUE_ALIASES:
        raise ValueError(
            f"Invalid technique: {technique}. "
            "Use zero-shot, one-shot, or few-shot."
        )

    normalized_techniques.append(TECHNIQUE_ALIASES[normalized_input])

TECHNIQUES = normalized_techniques

print("Configured techniques:", TECHNIQUES)


## 9. Hugging Face Token and Local Model Loading


In [ ]:
# On RunPod, set the token as the HF_TOKEN environment variable.
# For public models or models already stored locally, it can be None.
#
# Example in the RunPod terminal:
# export HF_TOKEN="hf_..."

HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    print("HF_TOKEN found. It will be used only to download the model.")
else:
    print(
        "HF_TOKEN not found. This works only for public models "
        "or models already stored locally."
    )


In [ ]:
if USE_4BIT and not torch.cuda.is_available():
    raise RuntimeError("4-bit quantization requires a CUDA GPU.")

if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
else:
    COMPUTE_DTYPE = torch.float32

quantization_config = None

if USE_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

print("Downloading/loading tokenizer:", MODEL_ID)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=TRUST_REMOTE_CODE,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "device_map": "auto",
    "token": HF_TOKEN,
    "trust_remote_code": TRUST_REMOTE_CODE,
    "low_cpu_mem_usage": True,
}

if quantization_config is not None:
    model_kwargs["quantization_config"] = quantization_config
    model_kwargs["torch_dtype"] = COMPUTE_DTYPE
else:
    model_kwargs["torch_dtype"] = COMPUTE_DTYPE

print("Downloading/loading model weights:", MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    **model_kwargs,
)

model.eval()

print("Model loaded locally.")
print("Compute dtype:", COMPUTE_DTYPE)
print("Device map:", getattr(model, "hf_device_map", "single device"))

## 10. Reading, Generation, and Validation Functions


In [ ]:
def read_spreadsheet(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        last_error = None

        for encoding in ("utf-8", "utf-8-sig", "latin-1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError as error:
                last_error = error

        raise last_error

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)

    raise ValueError("The input must be a CSV, XLSX, or XLS file.")


def normalize_source_id(value) -> str:
    if pd.isna(value):
        raise ValueError("A case with no value in the 'id' column was found.")

    try:
        numeric = float(value)
        if numeric.is_integer():
            return str(int(numeric))
    except (TypeError, ValueError):
        pass

    return str(value).strip()


def slugify(value: str, max_length: int = 80) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9._-]+", "-", value)
    value = value.strip("-")
    return value[:max_length] or "item"


def build_chat_prompt(user_prompt: str) -> str:
    system_prompt = (
        "Você converte títulos de casos de teste em cenários Gherkin. "
        "Sua resposta deve conter somente um Scenario em Gherkin, "
        "sem JSON, Feature, tags, Markdown, comentários ou explicações."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return (
        f"System:\n{system_prompt}\n\n"
        f"User:\n{user_prompt}\n\n"
        "Assistant:\n"
    )


def clean_gherkin(raw_text: str) -> str:
    if raw_text is None:
        raise ValueError("The model returned None.")

    text = str(raw_text).strip()

    # Remove Markdown fences if the model does not follow the prompt.
    text = re.sub(r"^```(?:gherkin|text)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # Keep only the content starting from the first Scenario.
    match = re.search(r"(?im)^\s*Scenario\s*:", text)

    if not match:
        raise ValueError("The response does not contain 'Scenario:'.")

    text = text[match.start():]

    cleaned_lines = []
    scenario_count = 0

    for raw_line in text.splitlines():
        line = raw_line.strip()

        if not line:
            continue

        if re.match(r"(?i)^Feature\s*:", line):
            continue

        if line.startswith("@"):
            continue

        if re.match(r"(?i)^Scenario\s*:", line):
            scenario_count += 1
            if scenario_count > 1:
                break
            cleaned_lines.append(re.sub(r"(?i)^Scenario\s*:", "Scenario:", line))
            continue

        if re.match(r"(?i)^Given\b", line):
            cleaned_lines.append("  " + re.sub(r"(?i)^Given\b", "Given", line))
            continue

        if re.match(r"(?i)^When\b", line):
            cleaned_lines.append("  " + re.sub(r"(?i)^When\b", "When", line))
            continue

        if re.match(r"(?i)^Then\b", line):
            cleaned_lines.append("  " + re.sub(r"(?i)^Then\b", "Then", line))
            continue

        if re.match(r"(?i)^And\b", line):
            cleaned_lines.append("    " + re.sub(r"(?i)^And\b", "And", line))
            continue

        # Stop when explanatory text appears after the scenario.
        if cleaned_lines:
            break

    result = "\n".join(cleaned_lines).strip()
    validate_gherkin(result)
    return result


def validate_gherkin(gherkin: str) -> None:
    required_patterns = {
        "Scenario": r"(?m)^Scenario\s*:",
        "Given": r"(?m)^\s{2}Given\b",
        "When": r"(?m)^\s{2}When\b",
        "Then": r"(?m)^\s{2}Then\b",
    }

    if len(re.findall(r"(?m)^Scenario\s*:", gherkin)) != 1:
        raise ValueError("The response must contain exactly one Scenario.")

    missing = [
        keyword
        for keyword, pattern in required_patterns.items()
        if not re.search(pattern, gherkin)
    ]

    if missing:
        raise ValueError(
            "The response does not contain the required keywords: "
            + ", ".join(missing)
        )

    forbidden_patterns = [
        r"(?im)^Feature\s*:",
        r"(?m)^@",
        r"```",
        r"(?im)^\s*\{",
    ]

    if any(re.search(pattern, gherkin) for pattern in forbidden_patterns):
        raise ValueError("The response contains forbidden elements.")


def generate_local_gherkin(
    test_case: str,
    execution_seed: int,
    prompt_template: str,
) -> str:
    user_prompt = prompt_template.format(test_case=test_case)
    formatted_prompt = build_chat_prompt(user_prompt)

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_device = model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    input_length = inputs["input_ids"].shape[-1]
    set_seed(execution_seed)

    do_sample = TEMPERATURE > 0

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": do_sample,
        "top_p": TOP_P,
        "repetition_penalty": 1.05,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.pad_token_id,
    }

    if do_sample:
        generation_kwargs["temperature"] = max(TEMPERATURE, 0.05)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            **generation_kwargs,
        )

    generated_tokens = outputs[0][input_length:]

    raw_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return clean_gherkin(raw_text)


def save_json_atomic(data: dict, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary_path.replace(output_path)

## 11. Spreadsheet Preparation


In [ ]:
dataframe = read_spreadsheet(INPUT_FILE)

required_columns = {"id", "name"}
missing_columns = required_columns - set(dataframe.columns)

if missing_columns:
    raise ValueError(
        "The spreadsheet does not contain the required columns: "
        + ", ".join(sorted(missing_columns))
    )

if dataframe["id"].isna().any():
    raise ValueError("The 'id' column contains empty values.")

if dataframe["id"].duplicated().any():
    duplicated = dataframe.loc[
        dataframe["id"].duplicated(keep=False),
        "id",
    ].tolist()
    raise ValueError(f"The 'id' column contains duplicate values: {duplicated[:10]}")

dataframe = dataframe.copy()
dataframe["source_id"] = dataframe["id"].apply(normalize_source_id)
dataframe["original_case"] = dataframe["name"].fillna("").astype(str).str.strip()

empty_names = dataframe["original_case"].eq("")

if empty_names.any():
    lines = (dataframe.index[empty_names] + 2).tolist()
    raise ValueError(
        f"Cases with no name were found on rows: {lines[:10]}"
    )

dataframe["source_line"] = dataframe.index + 2
dataframe["case_id"] = dataframe["source_id"].apply(
    lambda value: f"TC_{slugify(value)}"
)

if NUM_CASES is not None:
    if NUM_CASES > len(dataframe):
        raise ValueError(
            f"{NUM_CASES} cases were requested, "
            f"but the spreadsheet contains {len(dataframe)}."
        )
    selected_cases = dataframe.head(NUM_CASES).copy()
else:
    selected_cases = dataframe.copy()

model_slug = slugify(MODEL_ID.replace("/", "-"))

print("Cases in spreadsheet:", len(dataframe))
print("Selected cases:", len(selected_cases))
print("Executions per case and per technique:", NUM_EXECUTIONS)
print("Techniques:", len(TECHNIQUES))
print(
    "Maximum total generations:",
    len(selected_cases) * NUM_EXECUTIONS * len(TECHNIQUES),
)


## 12. Sequential Execution: Zero-shot → One-shot → Few-shot


In [ ]:
def prepare_experiment(technique: str):
    output_path = OUTPUT_DIR / (
        f"geracoes_gherkin_{model_slug}_{technique}.json"
    )

    if output_path.exists():
        with output_path.open("r", encoding="utf-8") as file:
            experiment = json.load(file)

        compatibility_fields = {
            "model": MODEL_ID,
            "technique": technique,
            "source_file": Path(INPUT_FILE).name,
        }

        for field, expected_value in compatibility_fields.items():
            if experiment.get(field) != expected_value:
                raise ValueError(
                    f"The existing JSON {output_path.name} uses "
                    f"{field}={experiment.get(field)!r}, "
                    f"but the current configuration uses {expected_value!r}."
                )

        experiment["number_of_executions"] = NUM_EXECUTIONS
        print(f"[{technique}] Existing JSON found; resuming execution.")

    else:
        experiment = {
            "model": MODEL_ID,
            "technique": technique,
            "number_of_executions": NUM_EXECUTIONS,
            "source_file": Path(INPUT_FILE).name,
            "cases": [],
        }

    case_lookup = {
        case["case_id"]: case
        for case in experiment["cases"]
    }

    for _, row in selected_cases.iterrows():
        case_id = row["case_id"]

        if case_id not in case_lookup:
            case_record = {
                "case_id": case_id,
                "source_id": row["source_id"],
                "source_line": int(row["source_line"]),
                "original_case": row["original_case"],
                "generations": [],
            }
            experiment["cases"].append(case_record)
            case_lookup[case_id] = case_record
        else:
            case_lookup[case_id]["source_id"] = row["source_id"]
            case_lookup[case_id]["source_line"] = int(row["source_line"])
            case_lookup[case_id]["original_case"] = row["original_case"]

    experiment["cases"].sort(key=lambda item: item["source_line"])
    save_json_atomic(experiment, output_path)

    return experiment, case_lookup, output_path


def run_technique(technique: str):
    prompt_template = PROMPTS[technique]
    experiment, case_lookup, output_path = prepare_experiment(technique)

    total_expected = len(selected_cases) * NUM_EXECUTIONS
    generated_now = 0
    skipped = 0
    failed = []

    progress = tqdm(
        total=total_expected,
        desc=f"{technique}",
    )

    for case_position, (_, row) in enumerate(selected_cases.iterrows()):
        case_id = row["case_id"]
        case_record = case_lookup[case_id]

        existing_generation_ids = {
            generation["generation_id"]
            for generation in case_record["generations"]
        }

        for execution in range(1, NUM_EXECUTIONS + 1):
            generation_id = (
                f"{case_id}__{model_slug}__"
                f"{technique}__exec-{execution:02d}"
            )

            if generation_id in existing_generation_ids:
                skipped += 1
                progress.update(1)
                continue

            last_error = None
            gherkin = None

            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    # The same seed is used for the same case/execution
                    # across all techniques. Thus, the technique changes, but the
                    # seed policy remains comparable.
                    seed = BASE_SEED + (case_position * 10_000) + execution

                    gherkin = generate_local_gherkin(
                        test_case=row["original_case"],
                        execution_seed=seed,
                        prompt_template=prompt_template,
                    )
                    break

                except Exception as error:
                    last_error = str(error)

                    if attempt < MAX_RETRIES:
                        time.sleep(2 ** (attempt - 1))

            if gherkin is None:
                failed.append({
                    "case_id": case_id,
                    "execution": execution,
                    "error": last_error,
                })
                progress.write(
                    f"Failure: {case_id}, execution {execution}: {last_error}"
                )
                progress.update(1)
                continue

            case_record["generations"].append({
                "generation_id": generation_id,
                "execution": execution,
                "gherkin": gherkin,
            })

            existing_generation_ids.add(generation_id)
            generated_now += 1
            progress.update(1)

        case_record["generations"].sort(
            key=lambda item: item["execution"]
        )

        experiment["cases"].sort(
            key=lambda item: item["source_line"]
        )

        # Save after each case to allow resuming after an interruption.
        save_json_atomic(experiment, output_path)

    progress.close()

    print(f"\n[{technique}] completed.")
    print("New generations:", generated_now)
    print("Existing generations skipped:", skipped)
    print("Failures:", len(failed))
    print("JSON:", output_path)

    if failed:
        print("\nFirst failures:")
        for failure in failed[:20]:
            print(failure)

    return output_path


OUTPUT_PATHS = []

for technique in TECHNIQUES:
    print("\n" + "=" * 80)
    print("STARTING TECHNIQUE:", technique)
    print("=" * 80)

    output_path = run_technique(technique)
    OUTPUT_PATHS.append(output_path)

print("\nAll techniques have been processed.")


## 13. Verification of the Three JSON Files


In [ ]:
for output_path in OUTPUT_PATHS:
    with output_path.open("r", encoding="utf-8") as file:
        result = json.load(file)

    total_generations = sum(
        len(case["generations"])
        for case in result["cases"]
    )

    print("\nFile:", output_path.name)
    print("Model:", result["model"])
    print("Technique:", result["technique"])
    print("Configured executions:", result["number_of_executions"])
    print("Recorded cases:", len(result["cases"]))
    print("Recorded generations:", total_generations)


## Structure of Each JSON File

Each technique generates its own file:

```text
/workspace/resultados_gherkin/
├── geracoes_gherkin_<modelo>_zero-shot.json
├── geracoes_gherkin_<modelo>_one-shot.json
└── geracoes_gherkin_<modelo>_few-shot.json
```

Each file preserves the same structure as the original notebook, with the `technique`
field corresponding to the executed technique.


## 14. Compress the Three JSON Files (Optional)


In [ ]:
import shutil

ZIP_BASE = OUTPUT_DIR / "geracoes_gherkin_3_tecnicas"

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("JSON files available individually at:", OUTPUT_DIR)
for path in OUTPUT_PATHS:
    print(" -", path)

print("ZIP file with results:", zip_path)
